# 🦺 **YOLOv8 APD Detection**

> **Capstone Project A2 Kelompok 11 FILKOM**  
> **Sistem Monitoring K3 Smart-Factory Berbasis Computer Vision**  
> Studi kasus: **PT. Indonesia Epson Industry**

Notebook ini berisi dokumentasi proses training model deteksi APD menggunakan **YOLOv8**.  
Model difokuskan pada **3 class utama** yang dibutuhkan untuk pengembangan *violation logic*:

| No | Class | Fungsi dalam Sistem |
|---|---|---|
| 0 | `person` | Mendeteksi keberadaan pekerja sebagai objek utama |
| 1 | `helmet` | Mengecek penggunaan helm keselamatan |
| 2 | `vest` | Mengecek penggunaan rompi keselamatan |

🎯 **Tujuan utama training:** menghasilkan model `best.pt` yang dapat digunakan untuk proses deteksi APD secara real-time dan nantinya diintegrasikan dengan **OpenCV**, **backend**, serta **dashboard monitoring K3**.

## 📌 **1. Project Context**

Pada sistem monitoring K3, model computer vision dilakukan training pada 3 class utama:

- ***person*** sebagai objek utama yang diamati.
- ***helmet*** sebagai indikator penggunaan helm keselamatan.
- ***vest*** sebagai indikator penggunaan rompi keselamatan.

Pemilihan 3 class ini dilakukan agar model lebih ringan, lebih fokus, dan lebih sesuai dengan kebutuhan awal sistem. Hasil deteksi dari model akan digunakan untuk membantu sistem menentukan apakah pekerja sudah memenuhi standar penggunaan APD atau berpotensi melakukan pelanggaran.

## 🔗 **2. Mount Google Drive**

Google Drive digunakan sebagai pusat penyimpanan dataset, hasil training, konfigurasi eksperimen, dan file model terbaik yang nantinya akan digunakan pada tahap inference.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## ⚙️ **3. Setup Path dan Konfigurasi Awal**

Pada tahap ini, seluruh path utama didefinisikan agar proses training lebih terstruktur dan mudah dipahami.

In [2]:
from pathlib import Path
import shutil
import yaml
from collections import Counter

# =========================
# Main Path Configuration
# =========================

DATASET_ZIP = Path("/content/drive/MyDrive/dataset/dataset_site_construction_safety.zip")
RAW_DIR = Path("/content/raw_dataset")
DATASET_ROOT = Path("/content/raw_dataset/dataset")

OUT_DIR = Path("/content/site_construction_safety_3class")
RUNS_DIR = Path("/content/drive/MyDrive/runs_apd")
PROJECT_NAME = "site_construction_3class"

TARGET_CLASSES = ["person", "helmet", "vest"]

print("📦 Dataset ZIP:", DATASET_ZIP)
print("📁 Raw dataset directory:", RAW_DIR)
print("🎯 Output 3-class dataset:", OUT_DIR)
print("💾 Training result directory:", RUNS_DIR / PROJECT_NAME)
print("🏷️ Target classes:", TARGET_CLASSES)

📦 Dataset ZIP: /content/drive/MyDrive/dataset/dataset_site_construction_safety.zip
📁 Raw dataset directory: /content/raw_dataset
🎯 Output 3-class dataset: /content/site_construction_safety_3class
💾 Training result directory: /content/drive/MyDrive/runs_apd/site_construction_3class
🏷️ Target classes: ['person', 'helmet', 'vest']


## 📦 **4. Ekstraksi Dataset**

Dataset utama disimpan dalam bentuk file ZIP di Google Drive. Pada tahap ini, file dataset tersebut diekstrak ke environment Google Colab agar seluruh gambar, label, dan file konfigurasi dapat digunakan dalam proses training YOLOv8.

In [3]:
# Clean previous extraction
shutil.rmtree(RAW_DIR, ignore_errors=True)

# Extract dataset
!unzip -q "$DATASET_ZIP" -d "$RAW_DIR"

print("✅ Dataset berhasil diekstrak.")
print("📁 Isi folder raw dataset:")
!find "$RAW_DIR" -maxdepth 3 -type f -name "data.yaml"

✅ Dataset berhasil diekstrak.
📁 Isi folder raw dataset:
/content/raw_dataset/dataset/data.yaml


In [4]:
print("DATASET_ZIP:", DATASET_ZIP)
print("RAW_DIR:", RAW_DIR)

DATASET_ZIP: /content/drive/MyDrive/dataset/dataset_site_construction_safety.zip
RAW_DIR: /content/raw_dataset


In [5]:
!ls -lh "$DATASET_ZIP"

-rw------- 1 root root 534M Apr 17 10:17 /content/drive/MyDrive/dataset/dataset_site_construction_safety.zip


In [6]:
!unzip -t "$DATASET_ZIP"

Streaming output truncated to the last 5000 lines.
    testing: dataset/train/labels/ppe_1062_jpg.rf.9e8395283cc2dd9ec5a453330571b6e0.txt   OK
    testing: dataset/train/labels/ppe_1062_jpg.rf.ad228dcf9498ef00d425c30760bbaa63.txt   OK
    testing: dataset/train/labels/ppe_1062_jpg.rf.d3c32f7db1cc4591793478f882ddf160.txt   OK
    testing: dataset/train/labels/ppe_1062_jpg.rf.e2966fe6ad6c4510ee221c2e66387d29.txt   OK
    testing: dataset/train/labels/ppe_1062_jpg.rf.e64956c469128cc2c50f348c9afa49f3.txt   OK
    testing: dataset/train/labels/ppe_1188_jpg.rf.40b1ad2134a4977f49b71682ed5b5a78.txt   OK
    testing: dataset/train/labels/ppe_1188_jpg.rf.48de9800afa703e3bf946ee505a15c19.txt   OK
    testing: dataset/train/labels/ppe_1188_jpg.rf.631a342dd17258e8e1b634e2b2fbc261.txt   OK
    testing: dataset/train/labels/ppe_1188_jpg.rf.8bc93c9a93e3beb1b9dca92bb074204e.txt   OK
    testing: dataset/train/labels/ppe_1188_jpg.rf.9b3a5243866665e201d3ff040538ec0a.txt   OK
    testing: dataset/train/la

## 🧾 **5. Membaca Dataset Asli**

File `data.yaml` digunakan untuk membaca daftar class asli dari dataset sebelum dilakukan filtering menjadi 3 class.

In [7]:
data_yaml_path = DATASET_ROOT / "data.yaml"

print("📄 data.yaml ditemukan:", data_yaml_path.exists())
print("📍 Path:", data_yaml_path)

with open(data_yaml_path, "r") as f:
    raw_yaml = yaml.safe_load(f)

raw_names = raw_yaml["names"]

# Roboflow/YOLO dataset can store names as list or dict
if isinstance(raw_names, dict):
    raw_names = [raw_names[k] for k in sorted(raw_names)]

print("\n🏷️ Class asli pada dataset:")
for idx, name in enumerate(raw_names):
    print(f"{idx}: {name}")

📄 data.yaml ditemukan: True
📍 Path: /content/raw_dataset/dataset/data.yaml

🏷️ Class asli pada dataset:
0: Gloves
1: Helmet
2: Non-Helmet
3: Person
4: Shoes
5: Vest
6: bare-arms


## ✂️ **6. Filtering Dataset menjadi 3 Class**

Pada tahap ini, dataset asli akan difilter agar hanya menggunakan tiga class utama yang dibutuhkan untuk sistem monitoring K3, yaitu

*   `person`
*   `helmet`
*   `vest`

Filtering dilakukan agar model lebih fokus pada objek yang berhubungan langsung dengan violation logic. Class `person` digunakan sebagai objek utama pekerja, sedangkan `helmet` dan `vest` digunakan sebagai indikator kepatuhan penggunaan APD.

In [8]:
TARGET_CLASSES = ["person", "helmet", "vest"]

def normalize_class_name(name):
    return str(name).strip().lower().replace("-", "_").replace(" ", "_")

wanted_classes = {
    "person": "person",
    "helmet": "helmet",
    "vest": "vest",
}

print("🎯 Target classes:")
for idx, cls in enumerate(TARGET_CLASSES):
    print(f"{idx}: {cls}")

🎯 Target classes:
0: person
1: helmet
2: vest


### 🔁 **Tahap 1. Menyesuaikan Class untuk Training**

Pada tahap ini, class yang akan digunakan dalam training ditentukan terlebih dahulu, yaitu `person`, `helmet`, dan `vest`. Setiap class kemudian diberi nomor sesuai format YOLO agar label pada dataset dapat dibaca dengan benar saat proses training berjalan.

In [9]:
old_to_new = {}

for old_idx, old_name in enumerate(raw_names):
    normalized_name = normalize_class_name(old_name)

    if normalized_name in wanted_classes:
        new_class_name = wanted_classes[normalized_name]
        new_idx = TARGET_CLASSES.index(new_class_name)
        old_to_new[old_idx] = new_idx

print("🔁 Mapping class lama ke class baru:")
for old_idx, new_idx in old_to_new.items():
    print(f"{old_idx} ({raw_names[old_idx]}) ➜ {new_idx} ({TARGET_CLASSES[new_idx]})")

🔁 Mapping class lama ke class baru:
1 (Helmet) ➜ 1 (helmet)
3 (Person) ➜ 0 (person)
5 (Vest) ➜ 2 (vest)


### 📂 **Tahap 2. Menyiapkan Folder Dataset Training**

Setelah class yang digunakan sudah ditentukan, tahap berikutnya adalah menyiapkan folder dataset yang akan dipakai untuk training. Dataset ini disusun ke dalam tiga bagian utama, yaitu `train`, `valid`, dan `test`, agar proses training dan evaluasi model dapat berjalan dengan lebih terstruktur.

In [10]:
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)

splits = ["train", "valid", "test"]

for split in splits:
    (OUT_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (OUT_DIR / split / "labels").mkdir(parents=True, exist_ok=True)

print("📁 Output folder ready:", OUT_DIR)

📁 Output folder ready: /content/site_construction_safety_3class


### 🏷️ **Tahap 3. Memilih Label yang Sesuai**

Pada tahap ini, label pada dataset dibaca satu per satu untuk mengambil data yang sesuai dengan class yang digunakan, yaitu `person`, `helmet`, dan `vest`. Label yang sesuai akan dipertahankan, sedangkan class lain tidak digunakan karena training difokuskan pada kebutuhan deteksi pekerja dan APD utama.

In [11]:
def find_matching_image(image_folder, label_stem):
    image_extensions = [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]

    for ext in image_extensions:
        image_file = image_folder / f"{label_stem}{ext}"
        if image_file.exists():
            return image_file

    return None

### 🖼️ **Tahap 4. Menyusun Gambar dan Label**

Setelah label yang sesuai ditemukan, gambar yang memiliki class target akan disalin ke folder dataset training. Proses ini memastikan bahwa hanya gambar dan label yang relevan dengan kebutuhan model yang digunakan dalam proses training YOLOv8.

In [12]:
def filter_yolo_split(split_name):
    img_in = DATASET_ROOT / split_name / "images"
    lbl_in = DATASET_ROOT / split_name / "labels"

    img_out = OUT_DIR / split_name / "images"
    lbl_out = OUT_DIR / split_name / "labels"

    image_count = 0
    label_count = 0
    class_counter = Counter()

    for label_file in lbl_in.glob("*.txt"):
        filtered_lines = []

        with open(label_file, "r") as f:
            for line in f:
                parts = line.strip().split()

                if len(parts) < 5:
                    continue

                old_class_id = int(float(parts[0]))

                if old_class_id in old_to_new:
                    new_class_id = old_to_new[old_class_id]
                    parts[0] = str(new_class_id)

                    filtered_lines.append(" ".join(parts))
                    class_counter[TARGET_CLASSES[new_class_id]] += 1

        if filtered_lines:
            output_label_file = lbl_out / label_file.name

            with open(output_label_file, "w") as f:
                f.write("\n".join(filtered_lines) + "\n")

            for ext in [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]:
                image_file = img_in / f"{label_file.stem}{ext}"

                if image_file.exists():
                    shutil.copy(image_file, img_out / image_file.name)
                    image_count += 1
                    break

            label_count += len(filtered_lines)

    return {
        "images": image_count,
        "labels": label_count,
        "class_distribution": dict(class_counter)
    }

### ✅ **Tahap 5. Menjalankan Proses Filtering Dataset**

Tahap ini menjalankan proses penyusunan dataset untuk tiga bagian, yaitu `train`, `valid`, dan `test`. Setelah proses selesai, sistem akan menampilkan jumlah gambar, jumlah label, dan distribusi class yang berhasil digunakan dalam dataset training.

In [13]:
dataset_summary = {}

for split in splits:
    dataset_summary[split] = filter_yolo_split(split)

print("✅ Dataset 3 class selesai dibuat di:", OUT_DIR)

for split, summary in dataset_summary.items():
    print(f"\n📌 {split.upper()}")
    print("Images:", summary["images"])
    print("Labels:", summary["labels"])
    print("Class distribution:", summary["class_distribution"])

✅ Dataset 3 class selesai dibuat di: /content/site_construction_safety_3class

📌 TRAIN
Images: 9693
Labels: 66013
Class distribution: {'vest': 24906, 'person': 28268, 'helmet': 12839}

📌 VALID
Images: 336
Labels: 2316
Class distribution: {'vest': 893, 'helmet': 446, 'person': 977}

📌 TEST
Images: 225
Labels: 1455
Class distribution: {'vest': 545, 'helmet': 284, 'person': 626}


### 📄 **Tahap 6. Membuat File Konfigurasi `data.yaml`**

File `data.yaml` dibuat sebagai konfigurasi utama untuk training YOLOv8. File ini berisi lokasi folder dataset, jumlah class, dan nama class yang digunakan. Dengan file ini, YOLOv8 dapat membaca dataset training dengan struktur yang benar.

In [14]:
new_yaml = {
    "train": str(OUT_DIR / "train" / "images"),
    "val": str(OUT_DIR / "valid" / "images"),
    "test": str(OUT_DIR / "test" / "images"),
    "nc": len(TARGET_CLASSES),
    "names": TARGET_CLASSES
}

with open(OUT_DIR / "data.yaml", "w") as f:
    yaml.safe_dump(new_yaml, f, sort_keys=False)

print("📄 data.yaml baru berhasil dibuat:")
print(OUT_DIR / "data.yaml")

📄 data.yaml baru berhasil dibuat:
/content/site_construction_safety_3class/data.yaml


### 📊 **Tahap 7. Mengecek Ringkasan Dataset**

Setelah dataset selesai disiapkan, tahap terakhir adalah mengecek kembali isi konfigurasi dan ringkasan dataset. Pengecekan ini dilakukan untuk memastikan bahwa class `person`, `helmet`, dan `vest` sudah terbaca dengan benar sebelum masuk ke tahap training model.

In [15]:
with open(OUT_DIR / "data.yaml", "r") as f:
    print(f.read())

train: /content/site_construction_safety_3class/train/images
val: /content/site_construction_safety_3class/valid/images
test: /content/site_construction_safety_3class/test/images
nc: 3
names:
- person
- helmet
- vest



## 📊 **7. Ringkasan Dataset 3 Class**

Bagian ini menampilkan jumlah gambar, jumlah label, dan distribusi class setelah dataset difilter menjadi 3 class.

In [16]:
print("📊 Ringkasan Dataset 3 Class\n")

for split, info in dataset_summary.items():
    print(f"===== {split.upper()} SET =====")
    print("Jumlah gambar :", info["images"])
    print("Jumlah label  :", info["labels"])
    print("Distribusi class:")
    for class_name in TARGET_CLASSES:
        print(f"  - {class_name}: {info['class_distribution'].get(class_name, 0)}")
    print()

📊 Ringkasan Dataset 3 Class

===== TRAIN SET =====
Jumlah gambar : 9693
Jumlah label  : 66013
Distribusi class:
  - person: 28268
  - helmet: 12839
  - vest: 24906

===== VALID SET =====
Jumlah gambar : 336
Jumlah label  : 2316
Distribusi class:
  - person: 977
  - helmet: 446
  - vest: 893

===== TEST SET =====
Jumlah gambar : 225
Jumlah label  : 1455
Distribusi class:
  - person: 626
  - helmet: 284
  - vest: 545



## 🧩 **8. Preview Konfigurasi `data.yaml`**

File `data.yaml` ini akan digunakan oleh YOLOv8 sebagai konfigurasi utama training.

In [17]:
with open(OUT_DIR / "data.yaml", "r") as f:
    print(f.read())

train: /content/site_construction_safety_3class/train/images
val: /content/site_construction_safety_3class/valid/images
test: /content/site_construction_safety_3class/test/images
nc: 3
names:
- person
- helmet
- vest



## 🚀 **9. Install Library YOLOv8**

Model dilatih menggunakan library **Ultralytics YOLOv8** karena ringan, cepat, dan cocok untuk kebutuhan object detection berbasis real-time monitoring.

In [18]:
!pip -q install -U ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.9 MB/s eta 0:00:00


## 🏋️ **10. Training Model YOLOv8**

Konfigurasi training yang digunakan:

| Parameter | Nilai | Keterangan |
|---|---:|---|
| Base model | `yolov8n.pt` | Model YOLOv8 Nano yang ringan untuk eksperimen awal |
| Epoch | `30` | Jumlah iterasi training |
| Image size | `640` | Ukuran input gambar |
| Batch size | `16` | Jumlah data yang diproses per batch |
| Dataset | `site_construction_safety_3class` | Dataset hasil filtering 3 class |
| Output | Google Drive | Agar hasil training tetap tersimpan |

📌 **Catatan:**  
YOLOv8 Nano dipilih karena lebih ringan dan lebih sesuai untuk pengembangan prototipe awal. Jika performa model masih kurang, pengembangan berikutnya dapat mencoba `yolov8s.pt` atau menambah dataset yang lebih representatif.

In [19]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(OUT_DIR / "data.yaml"),
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    project=str(RUNS_DIR),
    name=PROJECT_NAME
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.47 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/site_construction_safety_3class/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s

## 🏆 **11. Mengecek File Model Hasil Training**

Setelah training selesai, YOLOv8 akan menghasilkan beberapa file. File paling penting adalah `best.pt`, yaitu model dengan performa terbaik selama proses training.

In [20]:
best_path = RUNS_DIR / PROJECT_NAME / "weights" / "best.pt"
last_path = RUNS_DIR / PROJECT_NAME / "weights" / "last.pt"

print("🏆 best.pt:", best_path.exists(), best_path)
print("🧾 last.pt:", last_path.exists(), last_path)

if best_path.exists():
    print("\n✅ Model terbaik berhasil ditemukan dan siap digunakan untuk inference.")
else:
    print("\n⚠️ best.pt belum ditemukan. Pastikan proses training sudah selesai.")

🏆 best.pt: True /content/drive/MyDrive/runs_apd/site_construction_3class/weights/best.pt
🧾 last.pt: True /content/drive/MyDrive/runs_apd/site_construction_3class/weights/last.pt

✅ Model terbaik berhasil ditemukan dan siap digunakan untuk inference.


## 📈 **12. Validasi Model pada Test Set**

Tahap validasi dilakukan untuk mengevaluasi performa model pada data yang tidak digunakan saat training.  
Metrik utama yang diperhatikan adalah:

- **Precision**: seberapa tepat model ketika memberikan prediksi.
- **Recall**: seberapa banyak objek yang berhasil ditemukan oleh model.
- **mAP50**: rata-rata akurasi deteksi pada IoU threshold 0.50.
- **mAP50-95**: rata-rata akurasi pada berbagai threshold IoU.

In [21]:
best_model = YOLO(str(best_path))

metrics = best_model.val(
    data=str(OUT_DIR / "data.yaml"),
    split="test",
    device=0
)

print(metrics)

Ultralytics 8.4.47 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 703.1±253.0 MB/s, size: 30.2 KB)
val: Scanning /content/site_construction_safety_3class/test/labels... 225 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 225/225 1.8Kit/s 0.1s
val: New cache created: /content/site_construction_safety_3class/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 2.0it/s 7.6s
                   all        225       1455       0.97      0.938       0.98      0.812
                person        217        626      0.959      0.938      0.971      0.834
                helmet        173        284      0.974      0.936      0.982       0.75
                  vest        216        545      0.976      0.941      0.985      0.853
Speed: 5.7ms preprocess, 6.6ms inference

## 🖼️ **13. Uji Prediksi pada Sample Image**

Tahap ini digunakan untuk melihat contoh hasil deteksi model secara visual. Hasil prediksi akan disimpan ke folder output YOLOv8.

In [22]:
# Run prediction on test images
prediction_results = best_model.predict(
    source=str(OUT_DIR / "test" / "images"),
    conf=0.25,
    save=True,
    project=str(RUNS_DIR),
    name=f"{PROJECT_NAME}_sample_predictions"
)

print("✅ Sample prediction selesai.")
print("📁 Hasil prediksi tersimpan di:", RUNS_DIR / f"{PROJECT_NAME}_sample_predictions")


image 1/225 /content/site_construction_safety_3class/test/images/00000383_jpg.rf.fc23a3568fc6bf94dcf7f06f34cb370d.jpg: 640x640 7 persons, 3 helmets, 6 vests, 7.1ms
image 2/225 /content/site_construction_safety_3class/test/images/00004_jpg.rf.bbf6628e4df58091dc9b3b182adcfd21.jpg: 640x640 2 persons, 1 vest, 7.1ms
image 3/225 /content/site_construction_safety_3class/test/images/00020_jpg.rf.1bc9dd394851662dd1ae25b63f16408a.jpg: 640x640 3 persons, 3 vests, 7.0ms
image 4/225 /content/site_construction_safety_3class/test/images/00097_jpg.rf.fb94c365ffd061b33d49af4ca0cf7225.jpg: 640x640 4 persons, 1 helmet, 1 vest, 9.8ms
image 5/225 /content/site_construction_safety_3class/test/images/00098_jpg.rf.f7dc9924ce6a4948c2733b179f1078d1.jpg: 640x640 2 persons, 1 vest, 7.0ms
image 6/225 /content/site_construction_safety_3class/test/images/00103_jpg.rf.08c2375deec9ba2dfb67fca1d3319d46.jpg: 640x640 2 persons, 1 helmet, 2 vests, 7.9ms
image 7/225 /content/site_construction_safety_3class/test/images/001

## ✅ **14. Kesimpulan Training**

Berdasarkan proses training ini, model YOLOv8 berhasil disiapkan untuk mendeteksi 3 class utama, yaitu:

- `person`
- `helmet`
- `vest`

Model ini menjadi komponen penting dalam sistem monitoring K3 karena hasil deteksinya akan digunakan untuk mendukung proses identifikasi pelanggaran APD secara real-time.  
Tahap berikutnya adalah mengintegrasikan model `best.pt` dengan OpenCV, backend, database, dan dashboard monitoring agar sistem dapat berjalan sebagai prototipe yang utuh.
